# 29 — ML Engineer Coding: Top 5 Problems

**Target:** Staff/Senior ML screens at Shield AI, Vantor, Helsing, Striveworks,
Rebellion Defense, DeepNight, Harmattan. These shops ask different questions
than ServiceTitan — less custom-data-structure HackerRank, more "implement core
numerical primitives from scratch and justify your choices."

The 5 here are the canonical "show me you understand the math" problems:

| # | Problem | What it tests |
|---|---------|---------------|
| 1 | Softmax + cross-entropy (numerically stable) | log-sum-exp trick, gradient sanity |
| 2 | K-means from scratch | EM-style alternation, init strategies, convergence |
| 3 | Top-K with heap (streaming) | min-heap pattern for top-K |
| 4 | Rolling window statistics | online algorithms, Welford for variance |
| 5 | Scaled dot-product attention | the transformer primitive, end-to-end |

Each problem: statement → naive vs stable → solution → 10 test cases.


In [ ]:
# Common imports for the notebook
import numpy as np
import math
np.random.seed(42)


---
## Problem 1 — Numerically Stable Softmax + Cross-Entropy

> Implement `softmax(x)` and `cross_entropy(logits, target)` from scratch in numpy.
> Both must be numerically stable for inputs in `[-1e4, 1e4]`.

### Why this matters
Every ML engineer hits this. Naive `np.exp(x) / np.exp(x).sum()` overflows for
`x = [1000, 1001]`. Naive `-log(softmax)` underflows for confident predictions.
The interviewer wants to see the **log-sum-exp trick** and **fused log-softmax**.

### The trick

For any constant `c`:

```
softmax(x)_i = exp(x_i) / Σ exp(x_j)
             = exp(x_i - c) / Σ exp(x_j - c)
```

Picking `c = max(x)` guarantees the largest exponent is `exp(0) = 1`, so no overflow.

For cross-entropy, fuse `log(softmax(x))`:

```
log_softmax(x)_i = x_i - logsumexp(x)
                 = x_i - (c + log Σ exp(x_j - c))
```

This avoids ever computing `exp` of a large number AND avoids `log` of a tiny number.


In [ ]:
import numpy as np


def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax along `axis`."""
    x = np.asarray(x, dtype=np.float64)
    x_max = np.max(x, axis=axis, keepdims=True)
    shifted = x - x_max
    exp_x = np.exp(shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable log-softmax via log-sum-exp."""
    x = np.asarray(x, dtype=np.float64)
    x_max = np.max(x, axis=axis, keepdims=True)
    shifted = x - x_max
    return shifted - np.log(np.sum(np.exp(shifted), axis=axis, keepdims=True))


def cross_entropy(logits: np.ndarray, targets: np.ndarray) -> float:
    """Mean cross-entropy. logits: (N, C). targets: (N,) of class indices in [0, C).
    Uses log-softmax to avoid underflow.
    """
    logits = np.asarray(logits, dtype=np.float64)
    targets = np.asarray(targets, dtype=np.int64)
    if logits.ndim != 2:
        raise ValueError(f"logits must be 2D, got shape {logits.shape}")
    N, C = logits.shape
    if targets.shape != (N,):
        raise ValueError(f"targets shape {targets.shape} doesn't match N={N}")
    if (targets < 0).any() or (targets >= C).any():
        raise ValueError("target indices out of range")
    log_probs = log_softmax(logits, axis=1)
    # gather log_probs at target indices
    chosen = log_probs[np.arange(N), targets]
    return float(-chosen.mean())


In [ ]:
# ── Test cases for Problem 1 ──────────────────────────────────────────────

# 1. softmax sums to 1
out = softmax(np.array([1.0, 2.0, 3.0]))
assert abs(out.sum() - 1.0) < 1e-12

# 2. softmax invariant to additive shift (this IS the stability trick)
a = softmax(np.array([1.0, 2.0, 3.0]))
b = softmax(np.array([1001.0, 1002.0, 1003.0]))
assert np.allclose(a, b, atol=1e-12)

# 3. softmax of uniform inputs is uniform
out = softmax(np.zeros(7))
assert np.allclose(out, np.ones(7) / 7)

# 4. extreme positive — naive would overflow
out = softmax(np.array([1e4, 1e4 + 1]))
assert np.isfinite(out).all()
assert abs(out.sum() - 1.0) < 1e-12

# 5. extreme negative — naive denominator would be 0
out = softmax(np.array([-1e4, -1e4 - 1]))
assert np.isfinite(out).all()
assert abs(out.sum() - 1.0) < 1e-12

# 6. one-hot perfect prediction → cross-entropy ≈ 0
logits = np.array([[1e3, 0, 0]])
targets = np.array([0])
assert cross_entropy(logits, targets) < 1e-6

# 7. uniform prediction over C classes → CE = log(C)
C = 5
logits = np.zeros((1, C))
targets = np.array([2])
assert abs(cross_entropy(logits, targets) - math.log(C)) < 1e-12

# 8. confident WRONG prediction → CE is large but finite (no overflow)
logits = np.array([[1e3, 0, 0]])
targets = np.array([1])  # truth says class 1 but model says class 0
ce = cross_entropy(logits, targets)
assert np.isfinite(ce) and ce > 100  # large but not inf

# 9. batched: mean over N samples
logits = np.array([[1.0, 2.0, 3.0], [3.0, 2.0, 1.0]])
targets = np.array([2, 0])
# both are confident-correct → CE should be small
assert cross_entropy(logits, targets) < math.log(3)

# 10. invalid target index raises
try:
    cross_entropy(np.zeros((2, 3)), np.array([0, 5])); assert False
except ValueError:
    pass

# Bonus: log_softmax matches log(softmax) where the latter is computable
x = np.array([1.0, 2.0, 3.0])
assert np.allclose(log_softmax(x), np.log(softmax(x)))

print("✓ Problem 1 (Softmax + CE) — all 10 test cases pass")


---
## Problem 2 — K-means From Scratch

> Implement K-means clustering. Given `X` of shape `(N, D)` and `k`, return:
> - `labels` — array of length N with cluster assignments
> - `centroids` — array of shape `(k, D)`

### Algorithm

```
1. Initialize k centroids (random sample of X, or k-means++)
2. Repeat until convergence:
     a. Assign each point to nearest centroid (E-step)
     b. Update each centroid to mean of its assigned points (M-step)
3. Stop when assignments stop changing (or max_iter reached)
```

### Key design decisions

- **Initialization**: random init is unstable. K-means++ picks each new centroid
  with probability proportional to distance from the nearest existing one. Massively
  reduces the chance of bad local minima.
- **Empty clusters**: if a cluster gets zero points, re-initialize it to the
  farthest point from any centroid.
- **Convergence**: cheaper to check "no assignments changed" than "centroid drift < ε".


In [ ]:
import numpy as np
from typing import Tuple


def _kmeans_pp_init(X: np.ndarray, k: int, rng: np.random.Generator) -> np.ndarray:
    """K-means++ initialization."""
    N = X.shape[0]
    # pick first centroid uniformly
    centroids = [X[rng.integers(N)]]
    for _ in range(k - 1):
        # distance from each point to nearest existing centroid
        dists = np.min(
            np.linalg.norm(X[:, None, :] - np.array(centroids)[None, :, :], axis=2),
            axis=1,
        )
        probs = dists**2
        total = probs.sum()
        if total == 0:
            centroids.append(X[rng.integers(N)])
        else:
            probs = probs / total
            idx = rng.choice(N, p=probs)
            centroids.append(X[idx])
    return np.array(centroids)


def kmeans(
    X: np.ndarray,
    k: int,
    max_iter: int = 100,
    seed: int | None = 0,
) -> Tuple[np.ndarray, np.ndarray, int]:
    """K-means with k-means++ init. Returns (labels, centroids, n_iter)."""
    X = np.asarray(X, dtype=np.float64)
    if X.ndim != 2:
        raise ValueError(f"X must be 2D, got {X.shape}")
    N, D = X.shape
    if k <= 0 or k > N:
        raise ValueError(f"k must be in [1, N={N}], got {k}")

    rng = np.random.default_rng(seed)
    centroids = _kmeans_pp_init(X, k, rng)
    labels = np.full(N, -1, dtype=np.int64)

    for it in range(1, max_iter + 1):
        # E-step: assign each point to nearest centroid
        dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
        new_labels = np.argmin(dists, axis=1)

        if np.array_equal(new_labels, labels):
            return labels, centroids, it  # converged
        labels = new_labels

        # M-step: update centroids
        for c in range(k):
            mask = labels == c
            if mask.any():
                centroids[c] = X[mask].mean(axis=0)
            else:
                # empty cluster — re-init to point farthest from any centroid
                far_idx = np.argmax(np.min(dists, axis=1))
                centroids[c] = X[far_idx]

    return labels, centroids, max_iter


In [ ]:
# ── Test cases for Problem 2 ──────────────────────────────────────────────

# Helper: well-separated cluster generator
def _make_clusters(n_per, centers, noise=0.05, seed=0):
    rng = np.random.default_rng(seed)
    parts = [c + rng.normal(0, noise, (n_per, len(c))) for c in centers]
    X = np.vstack(parts)
    truth = np.repeat(np.arange(len(centers)), n_per)
    return X, truth


# 1. trivial — one cluster, all points there
X = np.random.randn(50, 2)
labels, _, _ = kmeans(X, k=1)
assert (labels == 0).all()

# 2. 2 well-separated clusters in 2D — recovered (up to label permutation)
X, truth = _make_clusters(50, [[0, 0], [10, 10]])
labels, centroids, _ = kmeans(X, k=2, seed=1)
# count agreement under best label assignment
from itertools import permutations
def best_acc(pred, truth, k):
    best = 0
    for perm in permutations(range(k)):
        remapped = np.array([perm[p] for p in pred])
        best = max(best, (remapped == truth).mean())
    return best
assert best_acc(labels, truth, 2) > 0.99

# 3. 3 clusters in 3D
X, truth = _make_clusters(40, [[0, 0, 0], [5, 0, 0], [0, 5, 0]])
labels, _, _ = kmeans(X, k=3, seed=1)
assert best_acc(labels, truth, 3) > 0.99

# 4. centroids approximately equal to true centers (after matching)
X, truth = _make_clusters(100, [[0, 0], [10, 10]], noise=0.1, seed=2)
_, centroids, _ = kmeans(X, k=2, seed=2)
# either centroid is near [0,0] and other near [10,10]
sorted_cs = sorted(centroids.tolist())
assert np.allclose(sorted_cs[0], [0, 0], atol=0.5)
assert np.allclose(sorted_cs[1], [10, 10], atol=0.5)

# 5. converges in well under max_iter for easy problem
X, _ = _make_clusters(50, [[0, 0], [10, 10]])
_, _, n_iter = kmeans(X, k=2, seed=1, max_iter=100)
assert n_iter < 20

# 6. invalid k raises
try:
    kmeans(np.zeros((5, 2)), k=0); assert False
except ValueError:
    pass
try:
    kmeans(np.zeros((5, 2)), k=10); assert False
except ValueError:
    pass

# 7. 1D data works
X = np.array([0, 1, 2, 100, 101, 102]).reshape(-1, 1).astype(float)
labels, _, _ = kmeans(X, k=2, seed=0)
# first 3 in one cluster, last 3 in other
assert labels[0] == labels[1] == labels[2]
assert labels[3] == labels[4] == labels[5]
assert labels[0] != labels[3]

# 8. deterministic given seed
X, _ = _make_clusters(50, [[0, 0], [10, 10]], seed=5)
l1, _, _ = kmeans(X, k=2, seed=42)
l2, _, _ = kmeans(X, k=2, seed=42)
assert np.array_equal(l1, l2)

# 9. different seeds may give same answer (well-separated) but should never crash
for s in range(5):
    labels, _, _ = kmeans(X, k=2, seed=s)
    assert best_acc(labels, np.repeat([0, 1], 50), 2) > 0.99

# 10. adversarial — duplicate points (degenerate distance matrix)
X = np.array([[0, 0]] * 10 + [[5, 5]] * 10, dtype=float)
labels, _, _ = kmeans(X, k=2, seed=0)
assert labels[:10].std() == 0 and labels[10:].std() == 0
assert labels[0] != labels[15]

print("✓ Problem 2 (K-means) — all 10 test cases pass")


---
## Problem 3 — Top-K Streaming with Min-Heap

> Given a stream of scores (you can't hold them all in memory), efficiently
> maintain the **top K** seen so far. Common in: retrieval re-ranking (keep top
> K relevant docs), recommendation (top K items per user), anomaly detection
> (top K outliers), beam search.

### Approaches

| Approach | Time per element | Space |
|----------|------------------|-------|
| Keep sorted list, insert each | O(K) per insert | O(K) |
| Keep min-heap of size K | **O(log K) per insert** | O(K) |
| Sort everything at end | O(N log N) and OOM | O(N) |

### The trick

Use a **min-heap of size K**. The smallest element is at the root. When a new
element arrives:
- If heap has fewer than K items, push it.
- Else if new item > heap[0] (root = current min of the top-K), replace and re-heapify.

At the end, the heap contains the K largest — in unspecified order. Sort for output.


In [ ]:
import heapq
from typing import Iterable, List, TypeVar

T = TypeVar("T")


class TopKHeap:
    """Streaming top-K maintainer using a min-heap of size K."""

    def __init__(self, k: int) -> None:
        if k <= 0:
            raise ValueError("k must be positive")
        self.k = k
        self._heap: list = []  # min-heap of size <= k

    def push(self, score: float, item: T = None) -> None:
        # store tuples for stable comparison; tiebreak on insertion order
        entry = (score, len(self._heap), item)
        if len(self._heap) < self.k:
            heapq.heappush(self._heap, entry)
        elif score > self._heap[0][0]:
            heapq.heapreplace(self._heap, entry)

    def push_many(self, scores_items: Iterable[tuple]) -> None:
        for s, it in scores_items:
            self.push(s, it)

    def top(self) -> List[tuple]:
        """Return current top-K, sorted descending by score."""
        return sorted([(s, it) for s, _, it in self._heap], key=lambda x: -x[0])

    def __len__(self) -> int:
        return len(self._heap)

    def __repr__(self) -> str:
        return f"TopKHeap(k={self.k}, size={len(self)})"


In [ ]:
# ── Test cases for Problem 3 ──────────────────────────────────────────────

# 1. simple top 3 of 10
t = TopKHeap(3)
for s in [5, 2, 8, 1, 9, 3, 7, 4, 6, 0]:
    t.push(s)
assert [s for s, _ in t.top()] == [9, 8, 7]

# 2. K larger than stream — return everything sorted
t = TopKHeap(10)
for s in [3, 1, 2]:
    t.push(s)
assert [s for s, _ in t.top()] == [3, 2, 1]

# 3. K=1 → just the max
t = TopKHeap(1)
for s in [5, 100, 3, 50]:
    t.push(s)
assert t.top()[0][0] == 100

# 4. with item payload preserved
t = TopKHeap(2)
t.push(0.9, "doc_a"); t.push(0.5, "doc_b"); t.push(0.8, "doc_c")
top = t.top()
assert [item for _, item in top] == ["doc_a", "doc_c"]

# 5. negative scores
t = TopKHeap(2)
for s in [-5, -1, -10, -3]:
    t.push(s)
assert [s for s, _ in t.top()] == [-1, -3]

# 6. ties — both get included
t = TopKHeap(3)
for s in [5, 5, 5, 5, 5]:
    t.push(s)
assert all(s == 5 for s, _ in t.top())
assert len(t) == 3

# 7. invalid k raises
try:
    TopKHeap(0); assert False
except ValueError:
    pass

# 8. push_many convenience
t = TopKHeap(3)
t.push_many([(0.1, "a"), (0.9, "b"), (0.5, "c"), (0.7, "d")])
assert [item for _, item in t.top()] == ["b", "d", "c"]

# 9. correctness vs naive sort on large stream
import random
random.seed(0)
stream = [random.random() for _ in range(10_000)]
t = TopKHeap(50)
for s in stream:
    t.push(s)
expected = sorted(stream, reverse=True)[:50]
got = [s for s, _ in t.top()]
assert got == expected

# 10. adversarial — strictly decreasing stream (worst case for naive append-and-sort)
t = TopKHeap(5)
for s in range(1000, 0, -1):
    t.push(s)
assert [s for s, _ in t.top()] == [1000, 999, 998, 997, 996]

print("✓ Problem 3 (Top-K Heap) — all 10 test cases pass")


---
## Problem 4 — Rolling Window Statistics (Online Algorithms)

> Implement `RollingStats(window_size)` supporting:
> - `add(x)` — observe a new value
> - `mean()`, `variance()`, `min()`, `max()` over the last `window_size` values
> - All operations O(1) or O(log W) amortized

### Why this matters
Anomaly detection, monitoring, signal smoothing. Naive `np.mean(buffer[-W:])` is O(W)
per call — too slow at 10k Hz sensor rates. The interviewer wants:
1. **Running sum** for O(1) mean (with floating-point error caveats)
2. **Welford's algorithm** for stable running variance
3. **Monotonic deque** for O(1) amortized min/max

### Welford's update

```
n += 1
delta = x - mean
mean += delta / n
M2 += delta * (x - mean)        # uses new mean
variance = M2 / (n - 1)
```

For a *sliding* window, we also subtract the contribution of the value falling
off the back. The textbook two-pass formula avoids catastrophic cancellation
that hits the naive `E[X²] - E[X]²` formula.


In [ ]:
from collections import deque
import math


class RollingStats:
    """Sliding-window stats with O(1) amortized mean/min/max, O(W) variance.

    Note: maintaining a numerically-stable *sliding* variance in true O(1) is
    surprisingly hard. We pay O(W) for variance and O(1) for everything else.
    """

    def __init__(self, window_size: int) -> None:
        if window_size <= 0:
            raise ValueError("window_size must be positive")
        self.W = window_size
        self._buf: deque = deque()
        self._sum: float = 0.0
        self._min_dq: deque = deque()   # stores (value, index) — monotonic increasing
        self._max_dq: deque = deque()   # stores (value, index) — monotonic decreasing
        self._idx: int = 0              # global insertion counter

    def add(self, x: float) -> None:
        x = float(x)
        # evict oldest if at capacity
        if len(self._buf) >= self.W:
            old = self._buf.popleft()
            self._sum -= old
            evict_idx = self._idx - self.W
            if self._min_dq and self._min_dq[0][1] == evict_idx:
                self._min_dq.popleft()
            if self._max_dq and self._max_dq[0][1] == evict_idx:
                self._max_dq.popleft()
        # add new
        self._buf.append(x)
        self._sum += x
        # maintain min-deque (increasing): pop from back while >= x
        while self._min_dq and self._min_dq[-1][0] >= x:
            self._min_dq.pop()
        self._min_dq.append((x, self._idx))
        # maintain max-deque (decreasing): pop from back while <= x
        while self._max_dq and self._max_dq[-1][0] <= x:
            self._max_dq.pop()
        self._max_dq.append((x, self._idx))
        self._idx += 1

    def mean(self) -> float:
        if not self._buf:
            raise ValueError("empty window")
        return self._sum / len(self._buf)

    def variance(self, ddof: int = 1) -> float:
        n = len(self._buf)
        if n - ddof <= 0:
            raise ValueError(f"need at least {ddof + 1} samples for variance(ddof={ddof})")
        m = self.mean()
        return sum((x - m) ** 2 for x in self._buf) / (n - ddof)

    def std(self, ddof: int = 1) -> float:
        return math.sqrt(self.variance(ddof=ddof))

    def min(self) -> float:
        if not self._min_dq:
            raise ValueError("empty window")
        return self._min_dq[0][0]

    def max(self) -> float:
        if not self._max_dq:
            raise ValueError("empty window")
        return self._max_dq[0][0]

    def __len__(self) -> int:
        return len(self._buf)


In [ ]:
# ── Test cases for Problem 4 ──────────────────────────────────────────────

# 1. basic mean of single value
r = RollingStats(5)
r.add(10)
assert r.mean() == 10

# 2. mean over partial window (not yet full)
r = RollingStats(5)
for x in [1, 2, 3]: r.add(x)
assert r.mean() == 2

# 3. mean over full window
r = RollingStats(3)
for x in [1, 2, 3, 4, 5]: r.add(x)   # window is [3, 4, 5]
assert r.mean() == 4

# 4. min and max track the sliding window correctly
r = RollingStats(3)
for x in [5, 1, 3, 7, 2]: r.add(x)   # window is [3, 7, 2]
assert r.min() == 2 and r.max() == 7

# 5. variance with single-sample window raises (ddof=1)
r = RollingStats(5)
r.add(10)
try:
    r.variance(); assert False
except ValueError:
    pass

# 6. variance matches numpy (with ddof=1)
import numpy as np
data = [3.0, 7.0, 2.0, 8.0, 5.0]
r = RollingStats(len(data))
for x in data: r.add(x)
assert abs(r.variance(ddof=1) - np.var(data, ddof=1)) < 1e-10

# 7. empty window queries raise
r = RollingStats(5)
for fn in [r.mean, r.min, r.max]:
    try:
        fn(); assert False
    except ValueError:
        pass

# 8. invalid window size
try:
    RollingStats(0); assert False
except ValueError:
    pass

# 9. monotonically increasing input — min should be the front of window
r = RollingStats(3)
for x in [1, 2, 3, 4, 5]: r.add(x)
assert r.min() == 3 and r.max() == 5

# 10. adversarial — many evictions, internal deques stay bounded
r = RollingStats(10)
for x in range(10_000):
    r.add(float(x))
assert len(r) == 10
# internal min/max deques bounded by window size too
assert len(r._min_dq) <= 10 and len(r._max_dq) <= 10
assert r.min() == 9990 and r.max() == 9999

print("✓ Problem 4 (Rolling Stats) — all 10 test cases pass")


---
## Problem 5 — Scaled Dot-Product Attention From Scratch

> Implement scaled dot-product attention:
>
> ```
> Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V
> ```
>
> Support an optional **causal mask** (for autoregressive language models) and
> **padding mask** (to ignore padding tokens).

### Why this matters
This is the canonical "do you actually understand transformers" question at
LLM-heavy shops. Every transformer-based system — GPT, BERT, ViT, the SAR-ATR
classifiers Seth has worked on — is built on this primitive.

### Shapes

```
Q: (batch, seq_q, d_k)
K: (batch, seq_k, d_k)
V: (batch, seq_k, d_v)
mask: (batch, seq_q, seq_k) — optional, True where ATTEND ALLOWED
output: (batch, seq_q, d_v)
attention_weights: (batch, seq_q, seq_k)
```

### Two gotchas interviewers watch for

1. **`sqrt(d_k)` scaling** — without it, the softmax saturates as `d_k` grows
   (dot products grow like sqrt(d_k)). Most candidates forget this.
2. **Masking with `-inf` BEFORE softmax** — adding `-inf` to disallowed positions
   makes them produce 0 after softmax. Common bug: applying mask after softmax
   (which leaves them non-zero and breaks the probability sum).


In [ ]:
import numpy as np


def scaled_dot_product_attention(
    Q: np.ndarray,
    K: np.ndarray,
    V: np.ndarray,
    mask: np.ndarray | None = None,
    causal: bool = False,
):
    """Scaled dot-product attention.

    Shapes:
        Q: (B, T_q, d_k)
        K: (B, T_k, d_k)
        V: (B, T_k, d_v)
        mask: (B, T_q, T_k) bool, True = attend allowed. Optional.
        causal: if True, apply lower-triangular mask (T_q must equal T_k).

    Returns: (output (B, T_q, d_v), attention_weights (B, T_q, T_k))
    """
    Q, K, V = (np.asarray(a, dtype=np.float64) for a in (Q, K, V))
    if Q.ndim != 3 or K.ndim != 3 or V.ndim != 3:
        raise ValueError("Q, K, V must all be 3D (batch, seq, dim)")
    B, T_q, d_k = Q.shape
    B_k, T_k, d_kk = K.shape
    B_v, T_v, d_v = V.shape
    if not (B == B_k == B_v) or d_k != d_kk or T_k != T_v:
        raise ValueError(f"shape mismatch: Q={Q.shape} K={K.shape} V={V.shape}")

    # raw scores
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)   # (B, T_q, T_k)

    # combine masks
    if causal:
        if T_q != T_k:
            raise ValueError("causal attention requires T_q == T_k")
        causal_mask = np.tril(np.ones((T_q, T_k), dtype=bool))[None, :, :]
        scores = np.where(causal_mask, scores, -np.inf)
    if mask is not None:
        if mask.shape != (B, T_q, T_k):
            raise ValueError(f"mask shape must be {(B, T_q, T_k)}, got {mask.shape}")
        scores = np.where(mask, scores, -np.inf)

    # numerically stable softmax along T_k
    scores_max = np.max(scores, axis=-1, keepdims=True)
    # if a row is fully masked, scores_max = -inf; replace to avoid nan
    scores_max = np.where(np.isfinite(scores_max), scores_max, 0.0)
    exp_scores = np.exp(scores - scores_max)
    denom = exp_scores.sum(axis=-1, keepdims=True)
    # avoid 0/0 → 0 for fully-masked rows
    weights = np.where(denom > 0, exp_scores / np.maximum(denom, 1e-30), 0.0)

    out = weights @ V
    return out, weights


In [ ]:
# ── Test cases for Problem 5 ──────────────────────────────────────────────

# 1. attention weights sum to 1 along last axis (unmasked)
np.random.seed(0)
B, T, D = 2, 4, 8
Q = np.random.randn(B, T, D)
K = np.random.randn(B, T, D)
V = np.random.randn(B, T, D)
out, w = scaled_dot_product_attention(Q, K, V)
assert np.allclose(w.sum(axis=-1), 1.0)

# 2. output shape matches Q's batch/seq and V's d_v
B, Tq, Tk, dk, dv = 2, 5, 7, 4, 3
Q = np.random.randn(B, Tq, dk)
K = np.random.randn(B, Tk, dk)
V = np.random.randn(B, Tk, dv)
out, w = scaled_dot_product_attention(Q, K, V)
assert out.shape == (B, Tq, dv)
assert w.shape == (B, Tq, Tk)

# 3. identity case: Q = K = V = identity rows → each query attends mostly to itself
B, T, D = 1, 4, 4
Q = K = V = np.eye(T)[None, :, :].astype(float)
out, w = scaled_dot_product_attention(Q, K, V)
# diagonal should be the largest weight in each row
for i in range(T):
    assert np.argmax(w[0, i]) == i

# 4. scaling actually divides by sqrt(d_k) — verify by comparing scales
B, T, D = 1, 2, 16
Q = K = np.ones((B, T, D))
V = np.eye(T)[None, :, :].astype(float)
_, w = scaled_dot_product_attention(Q, K, V)
# all Q·K^T are D (=16), scaled by sqrt(D)=4, then softmax → uniform
assert np.allclose(w, 1.0 / T)

# 5. causal mask blocks future positions
B, T, D = 1, 4, 4
Q = K = V = np.random.randn(B, T, D)
_, w = scaled_dot_product_attention(Q, K, V, causal=True)
# upper triangle (excluding diagonal) must be 0
upper = w[0]
for i in range(T):
    for j in range(i + 1, T):
        assert upper[i, j] == 0, f"future leak at ({i},{j})"

# 6. causal mask: row sums still equal 1
B, T, D = 1, 5, 4
Q = K = V = np.random.randn(B, T, D)
_, w = scaled_dot_product_attention(Q, K, V, causal=True)
assert np.allclose(w[0].sum(axis=-1), 1.0)

# 7. padding mask zeros out specified positions
B, Tq, Tk, D = 1, 2, 4, 4
Q = np.random.randn(B, Tq, D)
K = V = np.random.randn(B, Tk, D)
mask = np.ones((B, Tq, Tk), dtype=bool)
mask[0, :, 2:] = False   # mask out last 2 keys
_, w = scaled_dot_product_attention(Q, K, V, mask=mask)
assert (w[0, :, 2:] == 0).all()
assert np.allclose(w[0, :, :2].sum(axis=-1), 1.0)

# 8. shape mismatch raises
try:
    scaled_dot_product_attention(
        np.zeros((1, 2, 3)), np.zeros((1, 2, 4)), np.zeros((1, 2, 5))
    ); assert False
except ValueError:
    pass

# 9. causal with non-square attention raises
try:
    Q = np.zeros((1, 3, 4)); K = V = np.zeros((1, 5, 4))
    scaled_dot_product_attention(Q, K, V, causal=True); assert False
except ValueError:
    pass

# 10. numerical stability — large dot products don't produce nan/inf
B, T, D = 1, 4, 4
Q = K = V = np.ones((B, T, D)) * 100   # huge values
out, w = scaled_dot_product_attention(Q, K, V)
assert np.isfinite(out).all() and np.isfinite(w).all()
assert np.allclose(w.sum(axis=-1), 1.0)

print("✓ Problem 5 (Attention) — all 10 test cases pass")


---
## Final Sanity Check


In [ ]:
print("=" * 60)
print("Notebook 29 — ML Engineer Coding Top 5")
print("=" * 60)
print(f"  Problem 1 (Softmax + CE):         PASS")
print(f"  Problem 2 (K-means):              PASS")
print(f"  Problem 3 (Top-K Heap):           PASS")
print(f"  Problem 4 (Rolling Stats):        PASS")
print(f"  Problem 5 (Attention):            PASS")
print("=" * 60)
print("All 50 test cases passing.")
